In [2]:
%load_ext dotenv 
%dotenv 

In [105]:
##Loading an environmnet file with the above code

# What are we doing?

## Objectives 


* Build a data pipeline that downloads price data from the internet, stores it locally, transforms it into return data, and stores the feature set.
    - Getting the data.
    - Schemas and index in dask.

* Explore the parquet format.
    - Reading and writing parquet files.
    - Read datasets that are stored in distributed files.
    - Discuss dask vs pandas as a small example of big vs small data.
    
* Discuss the use of environment variables for settings.
* Discuss how to use Jupyter notebooks and source code concurrently. 
* Logging and using a standard logger.

## About the Data

+ We will download the prices for a list of stocks.
+ The source is Yahoo Finance and we will use the API provided by the library yfinance.


## Medallion Architecture

+ The architecture that we are thinking about is called Medallion by [DataBricks](https://www.databricks.com/glossary/medallion-architecture). It is an ELT type of thinking, although our data is well-structured.

![Medallion Architecture (DataBicks)](./images/02_medallion_architecture.png)

+ In our case, we would like to optimize the number of times that we download data from the internet. 
+ Ultimately, we will build a pipeline manager class that will help us control the process of obtaining and transforming our data.

![](./images/02_target_pipeline_manager.png)

# Download Data

Download the [Stock Market Dataset from Kaggle](https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset). Note that you may be required to register for a free account.

Extract all files into the directory: `./05_src/data/prices_csv/`

Your folder structure should include the following paths:

+ `05_src/data/prices_csv/etfs`
+ `05_src/data/prices_csv/stocks`


In [3]:
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.getenv('SRC_DIR'))

from utils.logger import get_logger
_logs = get_logger(__name__)

A few things to notice in the code chunk above:

+ Libraries are ordered from high-level to low-level libraries from the package manager (pip in this case, but could be conda, poetry, etc.)
+ The command `sys.path.append("../05_src/)` will add the `../05_src/` directory to the path in the Notebook's kernel. This way, we can use our modules as part of the notebook.
+ Local modules are imported at the end. 
+ The function `get_logger()` is called with `__name__` as recommended by the documentation.

Now, to load the historical price data for stocks and ETFs, we could use:

In [5]:
import random

stock_files = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))

random.seed(42)
stock_files = random.sample(stock_files, 60)

dt_list = []
for s_file in stock_files:
    _logs.info(f"Reading file: {s_file}")
    dt = pd.read_csv(s_file).assign(
        source = os.path.basename(s_file),
        ticker = os.path.basename(s_file).replace('.csv', ''),
        Date = lambda x: pd.to_datetime(x['Date'])
    )
    dt_list.append(dt)
stock_prices = pd.concat(dt_list, axis = 0, ignore_index = True)



2025-09-27 15:46:49,644, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\TNC.csv
2025-09-27 15:46:49,667, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CBB.csv
2025-09-27 15:46:49,683, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ALDX.csv
2025-09-27 15:46:49,689, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\GLADD.csv
2025-09-27 15:46:49,694, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\FIXX.csv
2025-09-27 15:46:49,698, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ETJ.csv
2025-09-27 15:46:49,709, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CMCTP.csv
2025-09-27 15:46:49,713, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\BWG.csv
2025-09-27 15:46:49,721, 4121356139.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\VIAC.csv
2025-09-27 15:46:49,7

Verify the structure of the `stock_prices` data:

In [6]:
stock_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239659 entries, 0 to 239658
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Date       239659 non-null  datetime64[ns]
 1   Open       239656 non-null  float64       
 2   High       239656 non-null  float64       
 3   Low        239656 non-null  float64       
 4   Close      239656 non-null  float64       
 5   Adj Close  239656 non-null  float64       
 6   Volume     239656 non-null  float64       
 7   source     239659 non-null  object        
 8   ticker     239659 non-null  object        
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 16.5+ MB


We can subset our ticker data set using standard indexing techniques. A good reference for this type of data manipulation is Panda's [Documentation](https://pandas.pydata.org/docs/user_guide/indexing.html#indexing-and-selecting-data) and [Cookbook](https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-selection).

From the subset data frame, select one column and convert to list.

In [7]:
select_tickers = stock_prices['ticker'].unique().tolist()
select_tickers

['TNC',
 'CBB',
 'ALDX',
 'GLADD',
 'FIXX',
 'ETJ',
 'CMCTP',
 'BWG',
 'VIAC',
 'REI',
 'BLPH',
 'SMG',
 'MOH',
 'AMH',
 'AMAL',
 'BPYPN',
 'ERH',
 'FAMI',
 'PFG',
 'SPXC',
 'ALL',
 'RTTR',
 'EARN',
 'ZIXI',
 'TSN',
 'WST',
 'REG',
 'MNK',
 'ESGR',
 'NGD',
 'SLRX',
 'GLW',
 'ACN',
 'CSSE',
 'WORK',
 'MOS',
 'IPWR',
 'GLUU',
 'CRMT',
 'EOLS',
 'INSU',
 'BWEN',
 'BPMX',
 'LH',
 'BRQS',
 'KALU',
 'ITCB',
 'SRE',
 'GAZ',
 'AQMS',
 'NPK',
 'QRHC',
 'CGEN',
 'LEVL',
 'BGS',
 'RIV',
 'GURE',
 'TEF',
 'SYNH',
 'KEY']

# Storing Data in CSV



+ We have some data. How do we store it?
+ We can compare two options, CSV and Parqruet, by measuring their performance:

    - Time to save.
    - Space required.

In [8]:
def get_dir_size(path='.'):
    '''Returns the total size of files contained in path.'''
    total = 0
    with os.scandir(path) as it:
        for entry in it:
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    return total

In [9]:
import time
import shutil

In [10]:
temp = os.getenv("TEMP_DATA")
csv_dir = os.path.join(temp, "csv")
shutil.rmtree(csv_dir, ignore_errors=True) # Remove all the files in a directory  if exist (cleanup utility)
stock_csv = os.path.join(csv_dir, "stock_px.csv")
os.makedirs(csv_dir, exist_ok=True)

In [11]:

start = time.time()
stock_prices.to_csv(stock_csv, index = False)
end = time.time()

_logs.info(f'Writing data ({stock_prices.shape}) to csv took {end - start} seconds.')
_logs.info(f'CSV file size { os.path.getsize(stock_csv)*1e-6 } MB')

2025-09-27 15:47:29,140, 473192941.py, 5, INFO, Writing data ((239659, 9)) to csv took 1.4221205711364746 seconds.
2025-09-27 15:47:29,140, 473192941.py, 6, INFO, CSV file size 26.618403999999998 MB


## Save Data to Parquet

### Dask 

We can work with with large data sets and parquet files. In fact, recent versions of pandas support pyarrow data types and future versions will require a pyarrow backend. The pyarrow library is an interface between Python and the Appache Arrow project. The [parquet data format](https://parquet.apache.org/) and [Arrow](https://arrow.apache.org/docs/python/parquet.html) are projects of the Apache Foundation.

However, Dask is much more than an interface to Arrow: Dask provides parallel and distributed computing on pandas-like dataframes. It is also relatively easy to use, bridging a gap between pandas and Spark. 

In [12]:
import dask.dataframe as dd

parquet_dir = os.path.join(temp, "parquet")
shutil.rmtree(parquet_dir, ignore_errors=True)
os.makedirs(parquet_dir, exist_ok=True)

In [13]:
px_dd = dd.from_pandas(stock_prices, npartitions = len(select_tickers))

start = time.time()
px_dd.to_parquet(parquet_dir, engine = "pyarrow")
end = time.time()

_logs.info(f'Writing dd ({stock_prices.shape}) to parquet took {end - start} seconds.')
_logs.info(f'Parquet file size { get_dir_size(parquet_dir)*1e-6 } MB')

2025-09-27 15:47:52,417, 817812245.py, 7, INFO, Writing dd ((239659, 9)) to parquet took 0.6282491683959961 seconds.
2025-09-27 15:47:52,419, 817812245.py, 8, INFO, Parquet file size 9.713384999999999 MB


### Parquet files and Dask Dataframes

+ Parquet files are immutable: once written, they cannot be modified.
+ Dask DataFrames are a useful implementation to manipulate data stored in parquets.
+ Parquet and Dask are not the same: parquet is a file format that can be accessed by many applications and programming languages (Python, R, PowerBI, etc.), while Dask is a package in Python to work with large datasets using distributed computation.
+ **Dask is not for everything** (see [Dask DataFrames Best Practices](https://docs.dask.org/en/stable/dataframe-best-practices.html)). 

    - Consider cases suchas small to large joins, where the small dataframe fits in memory, but the large one does not. 
    - If possible, use pandas: reduce, then use pandas.
    - Pandas performance tips apply to Dask.
    - Use the index: it is beneficial to have a well-defined index in Dask DataFrames, as it may speed up searching (filtering) the data. A one-dimensional index is allowed.
    - Avoid (or minimize) full-data shuffling: indexing is an expensive operations. 
    - Some joins are more expensive than others. 

        * Not expensive:

            - Join a Dask DataFrame with a pandas DataFrame.
            - Join a Dask DataFrame with another Dask DataFrame of a single partition.
            - Join Dask DataFrames along their indexes.

        * Expensive:

            - Join Dask DataFrames along columns that are not their index.


# How do we store prices?

+ We can store our data as a single blob. This can be difficult to maintain, especially because parquet files are immutable.
+ Strategy: organize data files by ticker and date. Update only latest month.



In [19]:
# CLean up before start
PRICE_DATA = os.getenv("PRICE_DATA") 
import shutil              
if os.path.exists(PRICE_DATA):  
    shutil.rmtree(PRICE_DATA)
    

PermissionError: [WinError 5] Access is denied: '../../05_src/data/prices/ACN\\ACN_2001'

In [ ]:
# CLean up before start
#PRICE_DATA = os.getenv("PRICE_DATA") # to load an enviroment variable use os.getenv, created a variable called PRiCE_DATA 
#import shutil               # clean up 
#if os.path.exists(PRICE_DATA):  # if PRICE_DATA exists then we will remove it
#    shutil.rmtree(PRICE_DATA)

In [20]:
#added line
PRICE_DATA

'../../05_src/data/prices/'

In [79]:
stock_prices.columns #dataframe that was created yesterday(previous session) a bunch of cvs list files concantained one on top of the other. Here columns need to have the same identifier and same data type for when creating these concatenation.

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker'],
      dtype='object')

In [21]:
for ticker in stock_prices['ticker'].unique():
    ticker_dt = stock_prices[stock_prices['ticker'] == ticker]
    ticker_dt = ticker_dt.assign(Year = ticker_dt.Date.dt.year)
    for yr in ticker_dt['Year'].unique():
        yr_dd = dd.from_pandas(ticker_dt[ticker_dt['Year'] == yr],2)
        yr_path = os.path.join(PRICE_DATA, ticker, f"{ticker}_{yr}")
        os.makedirs(os.path.dirname(yr_path), exist_ok=True)
        yr_dd.to_parquet(yr_path, engine = "pyarrow")
    
    # for every ticker, create a folder. then, take all the data fof that ticker. And every year, save a couple of parquet files.

In [22]:
stock_prices['ticker'].unique()

array(['TNC', 'CBB', 'ALDX', 'GLADD', 'FIXX', 'ETJ', 'CMCTP', 'BWG',
       'VIAC', 'REI', 'BLPH', 'SMG', 'MOH', 'AMH', 'AMAL', 'BPYPN', 'ERH',
       'FAMI', 'PFG', 'SPXC', 'ALL', 'RTTR', 'EARN', 'ZIXI', 'TSN', 'WST',
       'REG', 'MNK', 'ESGR', 'NGD', 'SLRX', 'GLW', 'ACN', 'CSSE', 'WORK',
       'MOS', 'IPWR', 'GLUU', 'CRMT', 'EOLS', 'INSU', 'BWEN', 'BPMX',
       'LH', 'BRQS', 'KALU', 'ITCB', 'SRE', 'GAZ', 'AQMS', 'NPK', 'QRHC',
       'CGEN', 'LEVL', 'BGS', 'RIV', 'GURE', 'TEF', 'SYNH', 'KEY'],
      dtype=object)

In [23]:
# Added line from the session while explaining the previous code
stock_prices['ticker']=='TNC'

0          True
1          True
2          True
3          True
4          True
          ...  
239654    False
239655    False
239656    False
239657    False
239658    False
Name: ticker, Length: 239659, dtype: bool

In [24]:
# added line to get the stock_price of TNC ( where TNC's raw is true), from the session
stock_prices[stock_prices['ticker']=='TNC']

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker
0,1973-02-22,4.250000,4.250000,4.250000,4.250000,0.507389,800.0,TNC.csv,TNC
1,1973-02-23,4.250000,4.250000,4.250000,4.250000,0.507389,800.0,TNC.csv,TNC
2,1973-02-26,4.250000,4.250000,4.250000,4.250000,0.507389,800.0,TNC.csv,TNC
3,1973-02-27,4.250000,4.250000,4.250000,4.250000,0.507389,0.0,TNC.csv,TNC
4,1973-02-28,4.250000,4.250000,4.250000,4.250000,0.507389,0.0,TNC.csv,TNC
...,...,...,...,...,...,...,...,...,...
11878,2020-03-26,54.060001,57.869999,54.060001,57.610001,57.610001,103300.0,TNC.csv,TNC
11879,2020-03-27,55.000000,55.369999,52.320000,53.450001,53.450001,72800.0,TNC.csv,TNC
11880,2020-03-30,53.810001,58.240002,53.180000,58.020000,58.020000,92200.0,TNC.csv,TNC
11881,2020-03-31,57.270000,59.790001,55.880001,57.950001,57.950001,136700.0,TNC.csv,TNC


In [25]:
# added line to get the stock_price of TNC ( where TNC's raw is true), from the session to get a year
ticker_dt = stock_prices[stock_prices['ticker']=='TNC']
ticker_dt.Date.dt.year

0        1973
1        1973
2        1973
3        1973
4        1973
         ... 
11878    2020
11879    2020
11880    2020
11881    2020
11882    2020
Name: Date, Length: 11883, dtype: int32

Why would we want to store data this way?

+ Easier to maintain. We do not update old data, only recent data.
+ We can also access all files as follows.

# Load, Transform and Save 

## Load

+ Parquet files can be read individually or as a collection.
+ `dd.read_parquet()` can take a list (collection) of files as input.
+ Use `glob` to get the collection of files.

In [26]:
from glob import glob

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

## Transform

+ This transformation step will create a *Features* data set. In our case, features will be stock returns (we obtained prices).
+ Dask dataframes work like pandas dataframes: in particular, we can perform groupby and apply operations.
+ Notice the use of [an anonymous (lambda) function](https://realpython.com/python-lambda/) in the apply statement.

In [86]:
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)

C:\Users\AlexandreT2\AppData\Local\Temp\ipykernel_12364\102405636.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [87]:
dd_rets = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

## Lazy Exection

What does `dd_rets` contain?

In [88]:
dd_rets

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
npartitions=60,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,object,int32,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...


+ Dask is a lazy execution framework: commands will not execute until they are required. 
+ To trigger an execution in dask use `.compute()`.

In [89]:
dd_rets.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
ticker,,,,,,,,,,,
ACN,2002-01-02,26.900000,26.910000,25.950001,26.209999,19.703959,900500.0,ACN.csv,2002,NaN,NaN
ACN,2002-01-03,26.230000,26.270000,25.299999,25.389999,19.087507,698200.0,ACN.csv,2002,26.209999,-0.031286
ACN,2002-01-04,25.389999,28.200001,25.240000,27.700001,20.824106,2708500.0,ACN.csv,2002,25.389999,0.090981
ACN,2002-01-07,27.770000,27.799999,26.450001,26.450001,19.884382,1113600.0,ACN.csv,2002,27.700001,-0.045126
ACN,2002-01-08,26.459999,27.299999,24.120001,27.280001,20.508362,1746900.0,ACN.csv,2002,26.450001,0.031380
...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2020-02-11,7.640000,7.850000,7.320000,7.350000,7.350000,491700.0,ZIXI.csv,2020,7.610000,-0.034166
ZIXI,2020-02-12,7.380000,7.400000,7.110000,7.290000,7.290000,358600.0,ZIXI.csv,2020,7.350000,-0.008163
ZIXI,2020-02-13,7.230000,7.310000,7.150000,7.160000,7.160000,314500.0,ZIXI.csv,2020,7.290000,-0.017833


## Save

+ Apply transformations to calculate daily returns
+ Store the enriched data, the silver dataset, in a new directory.
+ Should we keep the same namespace? All columns?

In [91]:
# CLean up before save
FEATURES_DATA = os.getenv("FEATURES_DATA")
if os.path.exists(FEATURES_DATA):
    shutil.rmtree(FEATURES_DATA)
dd_rets.to_parquet(FEATURES_DATA, overwrite = True)

PermissionError: [WinError 5] Access is denied: '../../05_src/data/features/stock_features'

# Optional: from Jupyter to Command Line

+ We have drafted our code in a Jupyter Notebook. 
+ Finalized code should be written in Python modules.

## Object Oriented vs Functional Programming

+ We can use classes to keep parameters and functions together.
+ We *could* use Object Oriented Programming, but parallelization of data manipulation and modelling tasks benefit from *Functional Programming*.
+ An Idea: 

    - [Data Oriented Programming](https://blog.klipse.tech/dop/2022/06/22/principles-of-dop.html).
    - Use the class to bundle together parameters and functions.
    - Use stateless operations and treat all data objects as immutable (we do not modify them, we overwrite them).
    - Take advantage of [`@staticmethod`](https://realpython.com/instance-class-and-static-methods-demystified/).

The code is in `./05_src/stock_prices/data_manager.py`.

Our original design was:

![](./images/02_target_pipeline_manager.png)



In [ ]:
from stock_prices.data_manager import DataManager
dm = DataManager()

Download all prices.

In [ ]:
dm.process_sample_files()

Finally, add features to the data set and save to a *feature store*.

In [ ]:
dm.featurize()